# Serve SMI-TED via HTTP API

> Call the FastHTML JSON API at a **URL** to load any weight dir / finetune checkpoint, then embed.

First start the API (separate terminal):

```bash
cd max_ports && pixi run api
# → http://127.0.0.1:8080
```

| Method | Path | Purpose |
|--------|------|---------|
| `POST` | `/load` | Load pretrained dir or finetune `.pt` |
| `POST` | `/embeddings` | SMILES → vector(s) |
| `GET` | `/status` | Is MAX ready? |
| `POST` | `/stop` | Stop MAX Serve |


In [ ]:
from __future__ import annotations

import json
from typing import Any

import httpx

API = "http://127.0.0.1:8080"  # change if you used --host/--port


def api(method: str, path: str, **kwargs) -> Any:
    r = httpx.request(method, f"{API.rstrip('/')}{path}", timeout=300.0, **kwargs)
    if r.status_code >= 400:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()


print(api("GET", "/health"))
print(json.dumps(api("GET", "/status"), indent=2))

## Load pretrained weights

`weight_path` is relative to `max_ports/` (or absolute).

In [ ]:
status = api(
    "POST",
    "/load",
    json={
        "weight_path": "./model_assets/ibm-research_materials.smi-ted",
        "device": "cpu",  # or "gpu"
    },
)
print(json.dumps(status, indent=2))

## Or load a finetune checkpoint

Pass `checkpoint` + `task`. The API exports to `model_assets/smi-ted-{task}/` then serves.

In [ ]:
# Uncomment to switch to a finetune:
# status = api(
#     "POST",
#     "/load",
#     json={
#         "checkpoint": "./finetune_ckpts/esol/smoke_MODEL_STATE.pt",
#         "task": "esol",
#         "device": "cpu",
#     },
# )
# print(json.dumps(status, indent=2))
pass

## Embeddings / predictions

Same URL for both modes. Property mode also fills `predictions`.

In [ ]:
out = api("POST", "/embeddings", json={"smiles": ["CCO", "c1ccccc1", "CC(=O)O"]})
print("mode:", out["mode"], "task:", out["task"], "model:", out["model"])
for i, (s, v) in enumerate(zip(["CCO", "c1ccccc1", "CC(=O)O"], out["embeddings"])):
    if out.get("predictions") is not None:
        print(f"{s:12s}  prediction={out['predictions'][i]:.6f}")
    else:
        print(f"{s:12s}  dim={len(v)}  first3={[round(x, 4) for x in v[:3]]}")

## curl equivalents

```bash
curl -s http://127.0.0.1:8080/status | jq

curl -s -X POST http://127.0.0.1:8080/load \
  -H 'Content-Type: application/json' \
  -d '{"weight_path":"./model_assets/ibm-research_materials.smi-ted","device":"cpu"}' | jq

curl -s -X POST http://127.0.0.1:8080/embeddings \
  -H 'Content-Type: application/json' \
  -d '{"smiles":"CCO"}' | jq
```

In [ ]:
# api("POST", "/stop")  # optional: free GPU/CPU when done